[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/43_dapo_loss.ipynb)

# 🔴 困难：解耦裁剪与动态采样策略优化损失（DAPO）

实现 **解耦裁剪与动态采样策略优化（Decoupled Clip and Dynamic sAmpling Policy Optimization，DAPO）** 损失：这是面向长思维链（Long-CoT）推理大模型强化学习的核心优化目标，针对标准 GRPO 的熵坍缩、长回复梯度稀释与全对／全错无效组问题进行了三项关键改进。

### 1. 组内优势与动态采样过滤（Dynamic Sampling）
对每个提示词组 $g$，计算组内奖励均值 $\bar r_g$ 与总体标准差 $\text{std}_g = \sqrt{\frac{1}{K_g}\sum_{j \in g}(r_j - \bar r_g)^2}$（即 `unbiased=False`）：
- 若 $\text{std}_g = 0$（组内奖励完全相同，例如全对或全错），该组不提供有效梯度信号，将其标记为**无效组**（$v_i = 0$），**既不计入分子，也不计入分母的有效词元数**。
- 若 $\text{std}_g > 0$，标记 $v_i = 1$，并计算组内标准化优势：

$$
\hat A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}
$$

### 2. 解耦非对称裁剪（Clip-Higher）
对每个词元位置 $(i, t)$ 计算重要性采样比率 $r_{i,t} = \exp(\text{new\_logps}_{i,t} - \text{old\_logps}_{i,t})$，采用非对称上下界 $[1 - \varepsilon_{\text{low}},\; 1 + \varepsilon_{\text{high}}]$（通常调高上界如 $\varepsilon_{\text{high}} = 0.28 > \varepsilon_{\text{low}} = 0.2$，为低概率探索词元留出上调空间，防止熵坍缩）：

$$
s_{i,t} = \min\Big(r_{i,t} \hat A_i,\; \operatorname{clip}(r_{i,t},\, 1 - \varepsilon_{\text{low}},\, 1 + \varepsilon_{\text{high}})\,\hat A_i\Big)
$$

### 3. 词元级策略梯度归一化（Token-Level Loss）
令有效词元掩码为 $m'_{i,t} = m_{i,t} \cdot v_i$（未提供 `mask` 时 $m_{i,t} = 1$）。与 GRPO 先在序列内求均值再对样本求均值不同，DAPO 直接在**全批次所有有效词元**上归一化，消除对长推理链的惩罚：

$$
\mathcal{L}_{\text{DAPO}} = -\frac{\sum_{i=1}^{B} \sum_{t=1}^{T} m'_{i,t}\, s_{i,t}}{\sum_{i=1}^{B} \sum_{t=1}^{T} m'_{i,t}}
$$

实现注意事项：
- 将 `old_logps` 与优势 $\hat A_i$ 从计算图中分离，使梯度仅通过 `new_logps` 传播（且不引入额外 KL 惩罚项）。
- 若整批有效词元数 $\sum_{i,t} m'_{i,t} = 0$（如所有组方差均为 $0$），返回值为 $0$ 且保持对 `new_logps` 可微的标量张量（如 `new_logps.sum() * 0.0`）。

### 函数或类接口
```python
from torch import Tensor

def dapo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              clip_low: float = 0.2,
              clip_high: float = 0.28,
              eps: float = 1e-6) -> Tensor:
    """计算一个批次的 DAPO 词元级解耦裁剪策略梯度损失。

    new_logps: (B, T) 当前策略的词元级对数概率
    old_logps: (B, T) 旧策略（采样策略）的词元级对数概率
    rewards: (B,) 每条采样回复的标量奖励
    group_ids: (B,) 整数组编号，相同编号表示同一个提示词／组
    mask: 可选，形状为 (B, T) 的词元掩码，1 表示计入，0 表示忽略
    clip_low: 下裁剪偏移量 eps_low（下界为 1 - clip_low）
    clip_high: 上裁剪偏移量 eps_high（上界为 1 + clip_high）
    eps: 优势标准化分母中的数值稳定项
    返回：标量损失（Tensor）
    """
```

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor

In [ ]:
# ✏️ 在此编写你的实现

from torch import Tensor

def dapo_loss(new_logps: Tensor,
              old_logps: Tensor,
              rewards: Tensor,
              group_ids: Tensor,
              mask: Tensor | None = None,
              clip_low: float = 0.2,
              clip_high: float = 0.28,
              eps: float = 1e-6) -> Tensor:
    pass  # 动态采样过滤零方差组 + [1-clip_low, 1+clip_high] 非对称裁剪 + 全局有效词元归一化

In [ ]:
# 🧪 调试
ratios = torch.tensor([[1.25, 1.40], [0.75, 0.85]])
old_logps = torch.zeros(2, 2)
new_logps = ratios.log().requires_grad_(True)
rewards = torch.tensor([1.0, -1.0])
group_ids = torch.tensor([0, 0])
print('Loss:', dapo_loss(new_logps, old_logps, rewards, group_ids, eps=0.0).item())

In [ ]:
# ✅ 提交
from torch_judge import check
check('dapo_loss')